In [ ]:
import pandas as pd
import numpy as np
import joblib
import pandas as pd

from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

# Application of Best Model to new Data

## Functions to load new data - keep up to date with the ones used in testing notebook

the feature function must be up to date!!

In [ ]:
def get_targets(bikeData):
    bikeData = bikeData.copy()
    bikeData["target_1h"] = bikeData["BikeCount"].shift(-1)
    bikeData["target_24h"] = bikeData["BikeCount"].shift(-24)

    # remove rows where future target is unknown
    bikeData_model = bikeData.dropna().copy()
    return bikeData_model


def prepare_data(bikeData, target_col):
    # =====================================================
    # 3. Define X and y
    # =====================================================

    X = bikeData.drop(columns=[
        "target_1h",
        "target_24h"
    ])

    y = bikeData[target_col]

    # =====================================================
    # 4. One-hot encode categorical features
    # =====================================================

    categorical_cols = X.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    print("Before get_dummies:", X.shape)

    X = pd.get_dummies(
        X,
        columns=categorical_cols,
        drop_first=True
    )

    print("After get_dummies:", X.shape)
    return X, y


def create_features(bikeData):

    bikeData = bikeData.copy()

    # =====================================================
    # 1. Sort chronologically
    # =====================================================

    bikeData = bikeData.sort_values(
        ["Month", "Day", "Hour"]
    ).reset_index(drop=True)

    # =====================================================
    # 2. Lag features
    # =====================================================

    bikeData["lag_1h"] = bikeData["BikeCount"].shift(1)
    bikeData["lag_2h"] = bikeData["BikeCount"].shift(2)
    bikeData["lag_3h"] = bikeData["BikeCount"].shift(3)

    bikeData["lag_24h"] = bikeData["BikeCount"].shift(24)
    bikeData["lag_48h"] = bikeData["BikeCount"].shift(48)
    bikeData["lag_168h"] = bikeData["BikeCount"].shift(168)

    # =====================================================
    # 3. Rolling mean features
    # shift(1) prevents data leakage
    # =====================================================

    bikeData["rolling_mean_3h"] = (
        bikeData["BikeCount"].shift(1).rolling(3).mean()
    )

    bikeData["rolling_mean_6h"] = (
        bikeData["BikeCount"].shift(1).rolling(6).mean()
    )

    bikeData["rolling_mean_12h"] = (
        bikeData["BikeCount"].shift(1).rolling(12).mean()
    )

    bikeData["rolling_mean_24h"] = (
        bikeData["BikeCount"].shift(1).rolling(24).mean()
    )

    bikeData["rolling_mean_168h"] = (
        bikeData["BikeCount"].shift(1).rolling(168).mean()
    )

    # =====================================================
    # 4. Rolling min/max/std features
    # =====================================================

    bikeData["rolling_std_24h"] = (
        bikeData["BikeCount"].shift(1).rolling(24).std()
    )

    bikeData["rolling_min_24h"] = (
        bikeData["BikeCount"].shift(1).rolling(24).min()
    )

    bikeData["rolling_max_24h"] = (
        bikeData["BikeCount"].shift(1).rolling(24).max()
    )

    # =====================================================
    # 5. Difference / trend features
    # =====================================================

    bikeData["diff_1h"] = (
        bikeData["BikeCount"].shift(1) - bikeData["BikeCount"].shift(2)
    )

    bikeData["diff_24h"] = (
        bikeData["BikeCount"].shift(1) - bikeData["BikeCount"].shift(25)
    )


    # =====================================================
    # 7. Drop missing values
    # =====================================================

    bikeData_model = bikeData.dropna().copy()

    print("Original shape:", bikeData.shape)
    print("Model shape:   ", bikeData_model.shape)

    return bikeData_model



def get_data_for_testing(data):
    data = data.copy()
    data = create_features(data)
    data = get_targets(data)
    X, y = prepare_data(data, "target_1h")
    return X, y

## Retrain the best model on entire train data

In [ ]:
# Datei laden
bikeData = pd.read_excel("data/challenge_public_dataset.xlsx")

data_for_1h = bikeData.copy()
data_for_1h = create_features(data_for_1h)
data_for_1h = get_targets(data_for_1h)
X, y = prepare_data(data_for_1h, "target_1h")



# =====================================================
# Retrain model on ALL data
# using known best parameters
# =====================================================
## EXCHANGE TO USE THE TRUE BEST MODEL
model = Pipeline([
    ("model", XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
        # known best params
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05
    ))
])

model.fit(X, y)

print("Model retrained on full dataset.")

# =====================================================
# Save model
# =====================================================

joblib.dump(model, "best_model_1h.pkl")
    
print("Model saved.")

##### 24h horizon ########

data_for_24h = bikeData.copy()
data_for_24h = create_features(data_for_24h)
data_for_24h = get_targets(data_for_24h)
X, y = prepare_data(data_for_24h, "target_24h")


## EXCHANGE TO USE THE TRUE BEST MODEL
model = Pipeline([
    ("model", XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
        # known best params
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05
    ))
])

model.fit(X, y)

print("Model retrained on full dataset.")

# =====================================================
# Save model
# =====================================================

joblib.dump(model, "best_model_24h.pkl")
    
print("Model saved.")

## Test model on new data

In [ ]:
from sklearn.metrics import mean_squared_error

# Datei laden
bikeDataNew = pd.read_excel("PATH_TO_NEW_DATA.xlsx")


#### Model für 1h Vorhersage #### 
data_for_1h = bikeDataNew.copy()
X_1h, y_1h = get_data_for_testing(data_for_1h)

# =====================================================
# Reload model
# =====================================================

model_1h = joblib.load("best_model_1h.pkl")

print("Model reloaded.")

# Predict
y_pred_1h = model_1h.predict(X_1h)

# Evaluate
mse_1h = mean_squared_error(y_1h, y_pred_1h)

print(f"1h Forecast MSE: {mse_1h:.4f}")


#### Model für 24h Vorhersage #### 

#Load model
data_for_24h = bikeDataNew.copy()
X_24h, y_24h = get_data_for_testing(data_for_24h)

# Load trained model
model_24h = joblib.load("best_model_24h.pkl")

# Predict
y_pred_24h = model_24h.predict(X_24h)

# Evaluate
mse_24h = mean_squared_error(y_24h, y_pred_24h)

print(f"24h Forecast MSE: {mse_24h:.4f}")